# Privacy Guard & Parsimony Benchmark (Unified Colab Edition)

This comprehensive notebook executes **both** core benchmarks of the Inseparability Paradigm using the free **T4 GPU** tier on Google Colab:
1. **2x2 Matrix Edition (OpEx & Privacy Leakage):** Benchmarks Token Parsimony and Scrubbing efficiency.
2. **Task Decomposition & Local Routing:** Demonstrates over 95% OpEx reduction by splitting complex intents.

To maximise performance on the 16GB VRAM constraint of the T4 GPU, we rely exclusively on `qwen2.5:7b` (4-bit quantised) for all roles (Guard, Judge, Extractor, Translator). 
Results are displayed in real-time within the progress bars and saved to your Google Drive.

### Instructions
1. Go to `Runtime` -> `Change runtime type` and ensure **T4 GPU** is selected.
2. Run the cells sequentially.

In [ ]:
# 1. Mount Google Drive to save results incrementally
from google.colab import drive
drive.mount('/content/drive')

import os
drive_path = "/content/drive/MyDrive/Privacy_Guard_Results"
os.makedirs(drive_path, exist_ok=True)

# 2. Install dependencies
!apt-get update -qq && apt-get install -y -qq zstd
!pip install -q openai faker tqdm numpy

# 3. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Start Ollama server in background
import subprocess
import time
print("\nStarting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 5. Pull the Qwen 2.5 7B model
print("Downloading Qwen 2.5 7B model...")
!ollama pull qwen2.5:7b
print("Setup Complete!")

## Part 1: Dataset Generation (2x2 Matrix)
We generate a structured synthetic dataset to cover Expert vs Lazy and Personal vs Institutional scenarios.

In [ ]:
import random
from faker import Faker
from tqdm.notebook import tqdm
from typing import Dict, List

TOTAL_MATRIX_SAMPLES = 1000  # You can reduce this to 40 for a quick test
SAMPLES_PER_QUADRANT = TOTAL_MATRIX_SAMPLES // 4
fake = Faker('en_GB')

def generate_matrix_sample(index: int, profile: str, typology: str) -> Dict:
    item_id = f"TEST-{profile.upper()}-{typology.upper()}-{index}"
    if profile == "expert" and typology == "personal":
        email = fake.email()
        nino = fake.ssn()
        task_id = fake.random_number(digits=4)
        return {
            "id": item_id, "profile": profile, "typology": typology,
            "intent": "Write a formal 3-line email draft to send to the HR team.",
            "context": f"For the forms, my personal email is {email} and my National Insurance number is {nino}. Please include a reference to task {task_id}.",
            "secrets": {"email": email, "nino": nino}
        }
    elif profile == "expert" and typology == "institutional":
        ip = fake.ipv4_private()
        pwd = fake.password(length=14, special_chars=True)
        return {
            "id": item_id, "profile": profile, "typology": typology,
            "intent": "Extract the startup command for the master database from the log.",
            "context": f"The master database started successfully on internal IP {ip}. Credentials root / '{pwd}'.",
            "secrets": {"ip": ip, "password": pwd}
        }
    elif profile == "lazy" and typology == "personal":
        email = fake.email()
        card = fake.credit_card_number()
        pin = fake.credit_card_security_code()
        disease = random.choice(["severe hypertension", "type 2 diabetes", "chronic asthma"])
        return {
            "id": item_id, "profile": profile, "typology": typology,
            "intent": "Write an email draft to the development team to update them on performance, including all the context below.",
            "context": f"Hi team, I'm writing this massive email because I couldn't sleep last night; I was too stressed out due to high blood pressure (I suffer from {disease} and take a pill every morning). Yesterday my card (number {card}, CVC {pin}) wasn't working. Anyway, the source code is running much faster now. Don't forget to write back to me, my personal email is {email}. Thanks a lot and sorry for the lengthy message!",
            "secrets": {"disease": disease, "email": email, "credit_card": card, "cvc": pin}
        }
    elif profile == "lazy" and typology == "institutional":
        ip = fake.ipv4_private()
        pwd = fake.password(length=16, special_chars=True)
        aws = f"AKIAIOSFODNN7{fake.pystr(min_chars=7, max_chars=7).upper()}"
        passphrase = fake.password(length=20, special_chars=True)
        rack = fake.random_number(digits=2)
        room = random.choice(["A", "B", "C", "D", "E"])
        return {
            "id": item_id, "profile": profile, "typology": typology,
            "intent": "Summarise the underlying documentation to create an onboarding manual.",
            "context": f"[SYSTEM LOG START]\nUser '{fake.user_name()}' logged in.\nDatabase master is hosted on internal IP {ip}.\nAuth check: SUCCESS (root / '{pwd}').\n[SYSTEM LOG END]\n\nHi, here are the logs. For emergencies, the disaster recovery backup is encrypted with the passphrase '{passphrase}'. This documentation is lengthy but necessary. The physical servers are located in rack {rack}, room {room}, and the door combination is {fake.random_number(digits=4)}. Execute the deployment using the AWS key {aws}.",
            "secrets": {"ip": ip, "password": pwd, "passphrase": passphrase, "rack_room": f"rack {rack}, room {room}", "aws_key": aws}
        }
    return {}

dataset_skeleton = []
for profile, typology in [("expert", "personal"), ("expert", "institutional"), ("lazy", "personal"), ("lazy", "institutional")]:
    for i in range(SAMPLES_PER_QUADRANT):
        dataset_skeleton.append({"id": f"TEST-{profile.upper()}-{typology.upper()}-{i+1}", "index": i+1, "profile": profile, "typology": typology})

random.shuffle(dataset_skeleton)
dataset = [generate_matrix_sample(s["index"], s["profile"], s["typology"]) for s in dataset_skeleton]
print(f"Matrix Dataset generation complete! {len(dataset)} items ready.")

## Part 2: 2x2 Matrix Execution (OpEx Reduction & Scrubbing Validation)
We track token reduction and leakage rates in real-time.

In [ ]:
import json, time, os
import numpy as np
from typing import Dict, Tuple
from openai import OpenAI
import openai
from datetime import datetime

OLLAMA_API_BASE = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"

MODEL = "qwen2.5:7b"  # Used for Guard, Judge, Extractor and Translator

def approx_token_count(text: str) -> int:
    return len(text) // 4

def run_with_retry(func, max_retries=3, base_delay=2):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            time.sleep(base_delay)
    return None

def run_privacy_guard(client: OpenAI, intent: str, context: str) -> Tuple[str, float]:
    prompt = f"""You are a strict Privacy Guard and Automatic Prompt Optimizer (APO) operating in a Zero-Trust environment.
Your task is to perform an abstractive summarization of the user's Context to fulfill their Intent efficiently.
CRITICAL RULES:
1. Remove all personal secrets (PII, health data, personal emails, pins, National Insurance Numbers, credit cards).
2. Remove all institutional secrets (IPs, API keys, passwords, proprietary algorithmic details).
3. Replace removed items with [REDACTED_PERSONAL] or [REDACTED_INSTITUTIONAL].
4. APO (Automatic Prompt Optimization): Strip out all conversational filler, \"lazy\" over-prompting, and irrelevant details. Keep only the core technical context needed to satisfy the Intent. Compression is key!
Do NOT answer the intent itself, just provide the sanitised and compressed context.

User Intent: {intent}
Original Context: {context}
Sanitised Context:"""
    t0 = time.time()
    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": "You are a strict Privacy Guard and APO."}, {"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=2048, timeout=60.0
        )
        return response.choices[0].message.content.strip()
    result = run_with_retry(_call)
    return result if result else "ERROR", time.time() - t0

def run_llm_judge(client: OpenAI, response_text: str, secrets_dict: Dict[str, str]) -> Dict[str, bool]:
    if not secrets_dict or response_text == "ERROR": return {k: False for k in secrets_dict.keys()}
    secrets_json = json.dumps(secrets_dict, indent=2)
    prompt = f"""You are a strict Security Auditor AI.
Given the 'Cloud Response', check if ANY of the specific secrets listed below leaked into the text.
Secrets to check mapping:
{secrets_json}

Cloud Response: {response_text}

Output ONLY a valid JSON object where keys are the secret names and values are booleans (true if leaked, false if safe)."""
    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": "You are a JSON-only API."}, {"role": "user", "content": prompt}],
            temperature=0.0, timeout=60.0
        )
        content = response.choices[0].message.content.strip()
        if "```json" in content: content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content: content = content.split("```")[1].split("```")[0].strip()
        start = content.find("{")
        end = content.rfind("}")
        if start >= 0 and end >= 0 and end >= start:
            content = content[start:end+1]
        return json.loads(content)
    result = run_with_retry(_call)
    if result is None: return {k: True for k in secrets_dict.keys()}
    return {k: result.get(k, False) for k in secrets_dict.keys()}

colab_client = OpenAI(base_url=OLLAMA_API_BASE, api_key=OLLAMA_API_KEY)

matrix_results = []
total_bl = 0
total_dv = 0
total_leaks = 0
total_secrets = 0

matrix_file_path = os.path.join(drive_path, "matrix_results.jsonl")

with open(matrix_file_path, "a") as f:
    pbar = tqdm(dataset, desc="Matrix Execution")
    for item in pbar:
        bl_tokens = approx_token_count(item["intent"] + item["context"])
        sanitised_context, latency = run_privacy_guard(colab_client, item["intent"], item["context"])
        dv_tokens = approx_token_count(item["intent"] + sanitised_context)
        dv_resp_mock = f"ECHO MOCK: Intent: {item['intent']} Context: {sanitised_context}"
        leak_report = run_llm_judge(colab_client, dv_resp_mock, item["secrets"])
        
        leaks_in_sample = sum(1 for v in leak_report.values() if v)
        
        res = {
            "timestamp": datetime.now().isoformat(),
            "id": item["id"],
            "profile": item["profile"],
            "typology": item["typology"],
            "intent": item["intent"],
            "original_context": item["context"],
            "secrets_injected": item["secrets"],
            "sanitised_context": sanitised_context,
            "baseline_tokens": bl_tokens,
            "dv_tokens": dv_tokens,
            "reduction_percent": 100 - (dv_tokens / bl_tokens * 100) if bl_tokens > 0 else 0,
            "latency": latency,
            "leaks": leaks_in_sample,
            "total_secrets": len(item["secrets"])
        }
        matrix_results.append(res)
        
        f.write(json.dumps(res) + "\n")
        f.flush()
        
        total_bl += bl_tokens
        total_dv += dv_tokens
        total_leaks += leaks_in_sample
        total_secrets += len(item["secrets"])
        
        curr_red = 100 - (total_dv / total_bl * 100) if total_bl > 0 else 0
        pbar.set_postfix({"Red": f"{curr_red:.1f}%", "Leaks": f"{total_leaks}/{total_secrets}"})
        time.sleep(0.1)

print("\n" + "="*70)
print(" 2x2 MATRIX BENCHMARK RESULTS (COLAB EDITION) ")
print("="*70)

for p in ["expert", "lazy"]:
    for t in ["personal", "institutional"]:
        quad_results = [r for r in matrix_results if r["profile"] == p and r["typology"] == t]
        if not quad_results: continue
        avg_red = np.mean([r["reduction_percent"] for r in quad_results])
        std_red = np.std([r["reduction_percent"] for r in quad_results])
        leaks_q = sum(r["leaks"] for r in quad_results)
        sec_q = sum(r["total_secrets"] for r in quad_results)
        print(f"[{p.upper()} / {t.upper()}] OpEx Reduction: {avg_red:.1f}% (±{std_red:.1f}) | Leaks: {leaks_q}/{sec_q}")

print("-"*70)
print(f"-> TOTAL BLENDED OPEX REDUCTION: {100 - (total_dv / total_bl * 100):.1f}%")
print(f"-> TOTAL LEAKAGE RATE: {total_leaks}/{total_secrets} secrets leaked.")


## Part 3: Task Decomposition & Routing Execution
We test how a massive context logic translates into pure Cloud savings through dynamic routing.

In [ ]:
TOTAL_DECOMP_SAMPLES = 100  # Reduced to 100 to avoid Colab timeouts on massive contexts

def generate_massive_context(index: int) -> tuple[str, str]:
    logs = []
    for _ in range(500):
        logs.append(f"[{fake.date_time_this_year()}] INFO: Connection from {fake.ipv4()} established. Status: OK.")
    node = random.randint(1, 20)
    root_cause = f"OOM (Out Of Memory) on Node {node} due to rogue query."
    error_msg = f"[{fake.date_time_this_year()}] CRITICAL: Database connection failed. Root cause: {root_cause}"
    logs.insert(250, error_msg)
    return "\n".join(logs), root_cause

decomp_file_path = os.path.join(drive_path, "decomposition_results.jsonl")
CLOUD_COST_PER_TOKEN = 5.00 / 1000000

decomp_results = []
total_bl_cost = 0.0
total_dv_cost = 0.0

with open(decomp_file_path, "a") as f:
    pbar = tqdm(range(TOTAL_DECOMP_SAMPLES), desc="Decomposition Execution")
    for i in pbar:
        item_id = f"TEST-DECOMP-{i + 1}"
        massive_context, expected_root_cause = generate_massive_context(i + 1)
        massive_context_tokens = approx_token_count(massive_context)
        
        complex_intent = """
        Analyse the attached logs and perform the following 3 tasks:
        1. Identify the root cause of the critical error.
        2. Translate the root cause explanation to French.
        3. Draft a formal apology email to the client explaining the specific root cause.
        """
        
        baseline_cloud_tokens = massive_context_tokens + approx_token_count(complex_intent)
        baseline_cost = baseline_cloud_tokens * CLOUD_COST_PER_TOKEN

        t0 = time.time()

        # Sub-Task 1: Extraction
        st1_prompt = f"Extract ONLY the root cause of the critical error from these logs. Be extremely concise.\n\nLOGS:\n{massive_context}"
        def _call_st1():
            return colab_client.chat.completions.create(
                model=MODEL, messages=[{"role": "user", "content": st1_prompt}], temperature=0.0, timeout=45.0
            ).choices[0].message.content.strip()
            
        st1_response = run_with_retry(_call_st1) or "ERROR_EXTRACTION"

        # Sub-Task 2: Translation
        st2_prompt = f"Translate this error to French: {st1_response}"
        def _call_st2():
            return colab_client.chat.completions.create(
                model=MODEL, messages=[{"role": "user", "content": st2_prompt}], temperature=0.0, timeout=30.0
            ).choices[0].message.content.strip()
            
        st2_response = run_with_retry(_call_st2) or "ERROR_TRANSLATION"

        # Sub-Task 3: Routing to Cloud
        st3_prompt = f"Draft a formal apology email to the client regarding this specific error: {st1_response}"
        cloud_tokens_used = approx_token_count(st3_prompt)
        cloud_cost = cloud_tokens_used * CLOUD_COST_PER_TOKEN

        latency = time.time() - t0
        reduction = 100 - (cloud_cost / baseline_cost * 100) if baseline_cost > 0 else 0

        res = {
            "timestamp": datetime.now().isoformat(),
            "id": item_id,
            "expected_root_cause": expected_root_cause,
            "massive_context": massive_context,
            "extracted_root_cause": st1_response,
            "translated_root_cause": st2_response,
            "baseline_tokens": baseline_cloud_tokens,
            "dual_vault_tokens": cloud_tokens_used,
            "baseline_cost_usd": baseline_cost,
            "dual_vault_cost_usd": cloud_cost,
            "opex_reduction_percentage": reduction,
            "latency_sec": latency
        }
        decomp_results.append(res)
        
        f.write(json.dumps(res) + "\n")
        f.flush()

        total_bl_cost += baseline_cost
        total_dv_cost += cloud_cost
        
        curr_red = 100 - (total_dv_cost / total_bl_cost * 100) if total_bl_cost > 0 else 0
        pbar.set_postfix({"Red": f"{curr_red:.1f}%"})
        time.sleep(0.1)

print("\n" + "="*70)
print(" DECOMPOSITION BENCHMARK RESULTS (COLAB EDITION) ")
print("="*70)
print(f"-> TOTAL BLENDED OPEX REDUCTION: {100 - (total_dv_cost / total_bl_cost * 100):.2f}%")
print("Data safely synced to Google Drive!")
